# LangChain basics: prompts and chains

This notebook introduces the core LangChain pattern of combining prompt templates, chat models, and output parsers into chains.

The examples build a small restaurant menu assistant. The first section creates a single prompt-and-model chain. The second section composes several focused chains into a modern runnable sequence using LangChain Expression Language (LCEL).

## Environment setup

Load local environment variables, configure a chat model, and import the LangChain building blocks used throughout the notebook.

This notebook uses Groq because the rest of this repository already uses it for fast hosted inference. Add `GROQ_API_KEY` to your local `.env` file before running the model cells.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_groq import ChatGroq


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    msg = "Could not find project root containing pyproject.toml."
    raise FileNotFoundError(msg)


project_root = find_project_root(Path.cwd())
load_dotenv(project_root / ".env")

assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file."

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    reasoning_format="hidden",
)

output_parser = StrOutputParser()
llm

/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x147679b90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x147923e10>, model_name='qwen/qwen3-32b', temperature=1e-08, reasoning_format='hidden', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

## Basic prompt template

`PromptTemplate` turns named inputs into a formatted text prompt. Before calling a model, inspect the formatted prompt to confirm the variables land where you expect.

In [2]:
menu_prompt = PromptTemplate.from_template(
    """
You are helping a restaurant test concise menu copy.

Create one menu item for this request:
- Cuisine: {cuisine}
- Featured ingredient: {ingredient}
- Dining style: {dining_style}

Return exactly two lines:
Name: <short dish name>
Description: <one appetizing sentence, no more than 24 words>
""".strip()
)

menu_inputs = {
    "cuisine": "Korean",
    "ingredient": "mushrooms",
    "dining_style": "casual lunch",
}

formatted_prompt = menu_prompt.invoke(menu_inputs)
print(formatted_prompt.text)

You are helping a restaurant test concise menu copy.

Create one menu item for this request:
- Cuisine: Korean
- Featured ingredient: mushrooms
- Dining style: casual lunch

Return exactly two lines:
Name: <short dish name>
Description: <one appetizing sentence, no more than 24 words>


## Basic chain

A chain connects several runnable pieces. The pipe operator passes the output of each step into the next step.

Here, the prompt formats the input dictionary, the model generates a chat response, and `StrOutputParser` extracts the response text.

In [3]:
menu_chain = menu_prompt | llm | output_parser

menu_item = menu_chain.invoke(menu_inputs)
display(Markdown(menu_item))

Name: Truffle Mushroom Rice Bowl  
Description: Spicy-sweet gochujang truffle mushrooms over rice, topped with sesame seeds and pickled radish.

## Chat prompt template

`ChatPromptTemplate` is useful when you want role-specific messages instead of one plain text prompt. Chat prompts make system instructions and user requests explicit.

In [4]:
chat_menu_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a practical restaurant copywriter. Keep answers specific, concise, and plausible.",
        ),
        (
            "human",
            "Create one {dining_style} menu item for a {cuisine} restaurant featuring {ingredient}.",
        ),
    ]
)

chat_messages = chat_menu_prompt.invoke(menu_inputs)
for message in chat_messages.messages:
    print(f"{message.type.upper()}: {message.content}")

SYSTEM: You are a practical restaurant copywriter. Keep answers specific, concise, and plausible.
HUMAN: Create one casual lunch menu item for a Korean restaurant featuring mushrooms.


In [5]:
chat_menu_chain = chat_menu_prompt | llm | output_parser

chat_menu_item = chat_menu_chain.invoke(menu_inputs)
display(Markdown(chat_menu_item))

**Umami Mushroom Rice Bowl**  
Braised shiitake and king oyster mushrooms in a savory gochujang glaze, served over steamed rice with marinated tofu, spinach, and a sprinkle of sesame seeds. Served with a side of house-pickled kimchi.  

*Casual, hearty, and packed with earthy Korean flavors.*

## Sequence of chains

Current LangChain code usually expresses multi-step workflows with LCEL: build small runnable chains, then compose them while passing named intermediate outputs forward.

This sequence has three steps:

1. Generate a raw dish concept.
2. Turn the concept into polished menu copy.
3. Refine the copy for clarity and practical menu details.

In [6]:
dish_idea_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You invent grounded restaurant dish concepts. Avoid gimmicks and keep ideas feasible for a real kitchen.",
        ),
        (
            "human",
            "Suggest one {dining_style} dish concept for a {cuisine} restaurant featuring {ingredient}. Return 2-3 sentences.",
        ),
    ]
)

dish_idea_chain = dish_idea_prompt | llm | output_parser

dish_idea = dish_idea_chain.invoke(menu_inputs)
display(Markdown(dish_idea))

**Umami Mushroom Rice Bowl**  
A hearty bowl of short-grain rice topped with braised shiitake and oyster mushrooms in a savory-sweet gochujang-garlic glaze, garnished with pickled daikon, sesame seeds, and a soft-boiled egg. The mushrooms are caramelized for depth, balanced by a sprinkle of kimchi for tangy brightness. Simple, satisfying, and rooted in Korean comfort flavors.

In [7]:
menu_copy_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You convert dish concepts into polished restaurant menu copy.",
        ),
        (
            "human",
            """Write menu copy for this dish concept:

{dish_idea}

Return exactly two lines:
Name: <short dish name>
Description: <one sentence, no more than 24 words>""",
        ),
    ]
)

menu_copy_chain = menu_copy_prompt | llm | output_parser

menu_copy = menu_copy_chain.invoke({"dish_idea": dish_idea})
display(Markdown(menu_copy))

Name: Umami Mushroom Rice Bowl  
Description: Hearty short-grain rice bowl topped with braised shiitake and oyster mushrooms in savory-sweet gochujang-garlic glaze, pickled daikon, sesame seeds, and a soft-boiled egg, caramelized for depth, balanced by tangy kimchi—simple, satisfying Korean comfort.

In [8]:
refinement_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a senior menu editor. Improve clarity and appetite appeal without making unsupported claims.",
        ),
        (
            "human",
            """Refine this menu copy for a {dining_style} setting. Keep the same two-line format and add a short Note line if a likely allergen should be mentioned.

{menu_copy}""",
        ),
    ]
)

refinement_chain = refinement_prompt | llm | output_parser

final_menu_copy = refinement_chain.invoke(
    {
        "dining_style": menu_inputs["dining_style"],
        "menu_copy": menu_copy,
    }
)
display(Markdown(final_menu_copy))

**Umami Mushroom Rice Bowl**  
Hearty short-grain rice bowl with braised shiitake and oyster mushrooms in a savory-sweet gochujang-garlic glaze. Topped with pickled daikon, sesame seeds, a caramelized soft-boiled egg, and tangy kimchi for a comforting Korean-inspired dish.  

**Note:** Contains sesame.

## Compose the full sequence

The previous cells ran each chain manually so the intermediate outputs were visible.

Now compose the same workflow into one runnable sequence. `RunnablePassthrough.assign(...)` keeps the original inputs and adds new named fields as each step completes.

In [9]:
def pick_final_output(sequence_state: dict) -> str:
    """Return only the final menu copy from the sequence state."""
    return sequence_state["final_menu_copy"]


menu_sequence = (
    RunnablePassthrough.assign(dish_idea=dish_idea_chain)
    .assign(menu_copy=menu_copy_chain)
    .assign(final_menu_copy=refinement_chain)
    | RunnableLambda(pick_final_output)
)

sequence_result = menu_sequence.invoke(menu_inputs)
display(Markdown(sequence_result))

**Name:** Spicy Gochujang Mushroom Rice Bowl with Braised Shiitake or Oyster Mushrooms  
**Description:** Topped with a runny fried egg, pickled daikon, and sesame seeds—balanced heat and umami, perfect for vegetarians or meat lovers.  
**Note:** Contains sesame. Egg included.  

*Refinements:*  
- Simplified the name for clarity while retaining key flavor cues (gochujang, mushrooms).  
- Emphasized the "runny fried egg" as a textural highlight.  
- Streamlined the description to focus on balance and adaptability.  
- Added allergen note for sesame (common allergen) and egg (optional clarification).

## Inspect intermediate outputs from the sequence

When learning or debugging, return the full sequence state instead of only the final answer. This makes the data flow between chains explicit.

In [10]:
debug_menu_sequence = (
    RunnablePassthrough.assign(dish_idea=dish_idea_chain)
    .assign(menu_copy=menu_copy_chain)
    .assign(final_menu_copy=refinement_chain)
)

debug_result = debug_menu_sequence.invoke(menu_inputs)

for key in ["dish_idea", "menu_copy", "final_menu_copy"]:
    heading = key.replace("_", " ").title()
    display(Markdown(f"### {heading}\n\n{debug_result[key]}"))

### Dish Idea

**Umami Mushroom Rice Bowl**  
A hearty bowl of short-grain rice topped with braised shiitake and oyster mushrooms in a savory-sweet gochujang-garlic glaze, garnished with crispy kimchi, sesame seeds, and a soft-boiled egg. Served with a side of pickled radish for brightness. Simple, umami-rich, and balanced for a satisfying midday meal.

### Menu Copy

Name: Umami Mushroom Rice Bowl  
Description: Hearty short-grain rice bowl with braised shiitake and oyster mushrooms in a savory-sweet gochujang-garlic glaze, topped with crispy kimchi, sesame seeds, and a soft-boiled egg. Served with pickled radish for brightness—a umami-rich, balanced midday meal.

### Final Menu Copy

**Name:** Hearty Mushroom Rice Bowl  
**Description:** Short-grain rice bowl with braised shiitake and oyster mushrooms in a savory-sweet gochujang-garlic glaze, topped with crispy kimchi, sesame seeds, and a soft-boiled egg. Served with pickled radish for brightness—a satisfying lunch bursting with flavor.  
**Note:** Contains: egg, sesame.

## Summary

- `PromptTemplate` and `ChatPromptTemplate` convert named inputs into model-ready prompts.
- `prompt | llm | StrOutputParser()` builds a simple prompt-and-model chain with current LangChain APIs.
- Multi-step workflows are built by composing smaller runnables and passing named intermediate outputs forward.
- LCEL keeps each step easy to inspect while avoiding older chain abstractions.